In [0]:
CREATE CATALOG IF NOT EXISTS main;
CREATE SCHEMA IF NOT EXISTS main.bronze;
CREATE VOLUME IF NOT EXISTS main.bronze.raw_landing;

In [0]:
%python
#volume test
dbutils.fs.ls("/Volumes/main/bronze/raw_landing/")

In [0]:
%python
# reset python to refresh memory
dbutils.library.restartPython()

In [0]:
%python
# After dbutils.library.restartPython() (Cell 3), sys.path is reset and the
# `ingestion` package (in the project root, one level above notebooks/) is no
# longer importable. Re-add the project root to sys.path before importing.
# Any cell that imports from `ingestion` after a kernel restart needs this.
sys.path.insert(0, "/Workspace/Users/sanamica@gmail.com/cms-de-final-project")
# Invalidate the import cache — a previous failed import attempt (before
# sys.path was fixed) may have cached a negative result that blocks retries.

from ingestion.manifest import ensure_manifest_table
ensure_manifest_table(spark)

In [0]:
%python
%pip install httpx

In [0]:
%python
import httpx
print(httpx.__version__)

In [0]:
%python
import ingestion.config as c
from ingestion.run_ingest import main
print([s.key for s in c.DATASETS])
print([h.key for h in c.HOSPITAL_FILES])

In [0]:
%python
# ingests only the CMS datasets.
main(spark, hospital_files=[])

In [0]:
DESCRIBE TABLE main.bronze.ingestion_manifest

In [0]:
%python
# cheking ccn number
from ingestion.ccn_lookup import find_ccns
display(find_ccns(spark))

In [0]:
%python
# test internet access
from ingestion.check_internet_access import main
main()

In [0]:
%python
# checking one test dataset calls provider data catalog API.
#writes JSON into raw landing volume
from ingestion.run_ingest import main
main(spark)

In [0]:
%python
#run full ingetion
run_id = main(spark)

In [0]:
-- checking if files were skipped
  SELECT dataset_key, status, etag, last_modified, note
  FROM main.bronze.ingestion_manifest
  WHERE run_id = 'd70baaed-585e-41e9-8464-da0797e8f23c' AND source = 'hospital_price_file'
  ORDER BY event_ts

In [0]:
%python
# Reading 1st line of all 4 hospital mrf price files
import glob
for p in sorted(glob.glob("/Volumes/main/bronze/raw_landing/hospital_price_files/*/ingest_date=*/run_id=*/*")):
    print("\n=====", p.split("/")[-1])
    with open(p, "rb") as f:
        print(f.read(3000).decode("utf-8-sig", errors="ignore"))

In [0]:
%python
#installing ijson for vumc and sthss mrf
%pip install ijson

In [0]:
%python
from ingestion.bronze_load import load_all
load_all(spark)

In [0]:
%python
print(len(spark.table("main.bronze.wmc_mrf").columns))
display(spark.sql("DESCRIBE DETAIL main.bronze.wmc_mrf"))

In [0]:

SHOW TABLES IN main.bronze

In [0]:
SELECT raw_json FROM main.bronze.vumc_mrf LIMIT 1

In [0]:
%python
#checking name for St, Thomas hospital for specialty surgery
import glob
for p in sorted(glob.glob("/Volumes/main/bronze/raw_landing/hospital_price_files/*/ingest_date=*/run_id=*/*")):
    print("\n=====", p.split("/")[-1])
    with open(p, "rb") as f:
        print(f.read(3000).decode("utf-8-sig", errors="ignore"))
       


In [0]:
%python
import sys
import importlib
# After dbutils.library.restartPython() (Cell 5), sys.path is reset and the
# `ingestion` package (in the project root, one level above notebooks/) is no
# longer importable. Re-add the project root to sys.path before importing.
# Any cell that imports from `ingestion` after a kernel restart needs this.
sys.path.insert(0, "/Workspace/Users/sanamica@gmail.com/cms-de-final-project")
# Invalidate the import cache — a previous failed import attempt (before
# sys.path was fixed) may have cached a negative result that blocks retries.
importlib.invalidate_caches()

from ingestion.provider_data_catalog import list_datasets
import httpx

# Fetch the full dataset catalog from the CMS Provider Data Catalog API.
# list_datasets returns all datasets in one call (no pagination needed).
with httpx.Client() as client:
    datasets = list_datasets(client)

# Print the first 100 datasets to discover identifiers for config.py
for d in datasets[:100]:
    print(d.get("identifier"), "-", d.get("title"))

In [0]:
%python
with httpx.Client() as client:
    all_datasets = list_datasets(client)

for d in all_datasets:
    print(d.get("identifier"), "-", d.get("title"))



In [0]:
SELECT * FROM main.bronze.ingestion_manifest
WHERE run_id = (SELECT run_id FROM main.bronze.ingestion_manifest
                ORDER BY event_ts DESC LIMIT 1)
ORDER BY event_ts

In [0]:
-- check wether status is succeeded or start
  SELECT * FROM main.bronze.ingestion_manifest WHERE run_id = 'bf79c34e-9ece-4eb8-98a0-316681bc558d'


In [0]:
%python
# sanity check getting expected total from CMS need identiier
import httpx
with httpx.Client() as client:
    resp = client.get("https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items/xubh-q36u")
    print(resp.json())

In [0]:

-- actual total pulled in manifestation need run id
SELECT dataset_key, source, status, 
       SUM(row_count) AS total_rows_ingested,
       COUNT(*) AS total_pages
FROM main.bronze.ingestion_manifest
WHERE run_id = '22e0e1e1-d12d-46d8-8b67-c7e680fe6bdc'
  AND status = 'succeeded'
GROUP BY dataset_key, source, status

In [0]:
%python
from ingestion import config
from ingestion.run_ingest import main
main(spark, datasets=[], hospital_files=config.HOSPITAL_FILES)